In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
sys.path.insert(0, os.path.abspath('../../src'))
import fiona
import geopandas as gpd
from pathlib import Path
import gstools as gs
import json
import pandas as pd
import rasterio as rio
import processing_help

import warnings
warnings.filterwarnings('ignore', message='.*mindist parameter.*', category=FutureWarning)      # remove future warning of spline mindist since we aren't even providing it anyways

In [ ]:
os.getcwd()

In [ ]:
survey_characteristics = Path(os.getcwd()).parent.parent / 'analysis_data' / 'survey_characteristics.csv'

DATA_DIR = Path('/mnt/teamgroup/AutoDBM/source_data')
os.listdir(DATA_DIR)

validation_results = list(DATA_DIR.glob('*/*/*.csv'))
     

In [ ]:
survey_characteristics_df = pd.read_csv(survey_characteristics)
# survey_characteristics_df.survey.to_numpy()

In [ ]:
import numpy as np
rmse_agg = {}

for file in validation_results:

    if 'min' in str(file) or '15000' in str(file) or 'mean' in str(file) or 'nav_safe' in str(file):
        print(file)
        continue
    else:
        val_file = pd.read_csv(file)
        rmse_agg[f'{file.parts[-2]}/{file.stem.replace('_validation', '')}.tif'] = np.sqrt(np.nanmean((val_file.True_Depth.to_numpy() - val_file.Predicted_Depth.to_numpy())**2))
     

rmse_agg_df = (
    pd.DataFrame.from_dict(
        rmse_agg,
        orient="index",
        columns=["mean_rmse"],
    )
    .reset_index()
    .rename(columns={"index": "survey_surface"})
)

rmse_agg_df

In [ ]:
# Split survey_surface into survey name and method
rmse_agg_df[['survey', 'surface']] = rmse_agg_df['survey_surface'].str.split('/', expand=True)
rmse_agg_df['surface'] = rmse_agg_df['surface'].str.replace('_depth.tif', '', regex=False)

# Pivot to wide format: 1 row per survey, 1 column per method
rmse_wide = rmse_agg_df.pivot(index='survey', columns='surface', values='mean_rmse').reset_index()
rmse_wide.columns.name = None  # clean up the column axis name

rmse_wide.to_csv(Path(os.getcwd()).parent.parent / 'analysis_data' / 'aggregated_rmse.csv', index=False)